In [1]:
"""
=============================================================================
MODEL 7 — XGBoost Fusion Score Regressor with Monotone Constraints
=============================================================================
Predicts a continuous fusion threat score in [0, 1] from three domain signals
(CYBINT / SOCINT / GEOINT) plus a set of derived features.

Mathematical guarantee (monotone constraints):
  ∂fusion_score / ∂cybint_score  ≥ 0  always
  ∂fusion_score / ∂socint_score  ≥ 0  always
  ∂fusion_score / ∂geoint_score  ≥ 0  always
  ∂fusion_score / ∂domain_max   ≥ 0  always
  ∂fusion_score / ∂domain_min   ≥ 0  always
  ∂fusion_score / ∂mean_score   ≥ 0  always
  ∂fusion_score / ∂n_high_domains ≥ 0 always
  ∂fusion_score / ∂n_active_scenarios ≥ 0  always
  ∂fusion_score / ∂cyber_geo_interaction ≥ 0 always
  ∂fusion_score / ∂all_domains_elevated  ≥ 0 always

Threshold mapping (score × 100):
  score < 0.20          →  LOW      (label 0)
  0.20 ≤ score < 0.45   →  MODERATE (label 1)
  0.45 ≤ score < 0.85   →  HIGH     (label 2)
  score ≥ 0.85          →  CRITICAL (label 3)

Input data : fusion_clean_train.csv  (40 000 rows)
Target     : continuous fusion_score constructed from threat_label_int
             via band-constrained, per-label min-max normalisation of the
             weighted domain signal so that:
               - every LOW row has fusion_score ∈ [0.00, 0.20)
               - every MODERATE row has fusion_score ∈ [0.20, 0.45)
               - every HIGH row has fusion_score ∈ [0.45, 0.85)
               - every CRITICAL row has fusion_score ∈ [0.85, 1.00]
             Within each band the signal is monotone in the domain scores.
=============================================================================
"""

'\n=============================================================================\nMODEL 7 — XGBoost Fusion Score Regressor with Monotone Constraints\n=============================================================================\nPredicts a continuous fusion threat score in [0, 1] from three domain signals\n(CYBINT / SOCINT / GEOINT) plus a set of derived features.\n\nMathematical guarantee (monotone constraints):\n  ∂fusion_score / ∂cybint_score  ≥ 0  always\n  ∂fusion_score / ∂socint_score  ≥ 0  always\n  ∂fusion_score / ∂geoint_score  ≥ 0  always\n  ∂fusion_score / ∂domain_max   ≥ 0  always\n  ∂fusion_score / ∂domain_min   ≥ 0  always\n  ∂fusion_score / ∂mean_score   ≥ 0  always\n  ∂fusion_score / ∂n_high_domains ≥ 0 always\n  ∂fusion_score / ∂n_active_scenarios ≥ 0  always\n  ∂fusion_score / ∂cyber_geo_interaction ≥ 0 always\n  ∂fusion_score / ∂all_domains_elevated  ≥ 0 always\n\nThreshold mapping (score × 100):\n  score < 0.20          →  LOW      (label 0)\n  0.20 ≤ score < 0.45  

In [3]:
# ---------------------------------------------------------------------------
# 0.  Imports & global configuration
# ---------------------------------------------------------------------------
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for headless environments
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import xgboost as xgb
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

# ── Reproducibility ────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH  = "/Users/amitsingh/ML_Projects/WarfareAI/datasets/fusion_clean_train.csv"
MODEL_DIR  = "./outputs"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH  = os.path.join(MODEL_DIR, "model7_xgb_fusion.json")   # XGBoost native format
META_PATH   = os.path.join(MODEL_DIR, "model7_meta.joblib")        # feature list + thresholds
PLOT_PATH   = os.path.join(MODEL_DIR, "model7_diagnostics.png")

# ── Threat-level thresholds (applied to fusion_score directly, 0-1 scale) ─
THRESHOLDS = {
    "LOW":      (0.00, 0.20),   # score × 100 < 20
    "MODERATE": (0.20, 0.45),   # score × 100 ∈ [20, 45)
    "HIGH":     (0.45, 0.85),   # score × 100 ∈ [45, 85)
    "CRITICAL": (0.85, 1.01),   # score × 100 ≥ 85  (upper bound is open, 1.01 catches 1.0)
}
LABEL_TO_STR = {0: "LOW", 1: "MODERATE", 2: "HIGH", 3: "CRITICAL"}
STR_TO_LABEL = {v: k for k, v in LABEL_TO_STR.items()}

# ── XGBoost hyper-parameters ───────────────────────────────────────────────
XGB_PARAMS = {
    "objective":          "reg:squarederror",
    "n_estimators":       600,
    "learning_rate":      0.04,
    "max_depth":          5,
    "min_child_weight":   6,
    "subsample":          0.80,
    "colsample_bytree":   0.80,
    "gamma":              0.05,       # minimum loss reduction for a split
    "reg_alpha":          0.10,       # L1 regularisation
    "reg_lambda":         1.50,       # L2 regularisation
    "tree_method":        "hist",     # fast histogram method
    "device":             "cpu",
    "eval_metric":        ["rmse", "mae"],
    "random_state":       RANDOM_SEED,
    "n_jobs":             -1,
    "verbosity":          0,
    # ── Early stopping is handled externally via callbacks ─────────────────
}

# ── Monotone constraints (per-feature, matching FEATURE_COLS order) ────────
# +1  : fusion score MUST be non-decreasing as this feature increases
#  0  : unconstrained
# -1  : fusion score MUST be non-increasing (none here — all signals are either
#       positive or directionally ambiguous)
MONOTONE_CONSTRAINTS = {
    # ---- base domain signals -------------------------------------------
    "cybint_score":          +1,   # rising cyber threat → higher fused score
    "socint_score":          +1,   # rising social intel → higher fused score
    "geoint_score":          +1,   # rising geo threat  → higher fused score
    # ---- aggregate domain statistics -----------------------------------
    "domain_max":            +1,   # strongest domain drives the ensemble up
    "domain_min":            +1,   # even the weakest domain being higher → higher fused
    "domain_std":             0,   # spread is ambiguous (high std may mean only 1 domain hot)
    "domain_range":           0,   # same reasoning as std — directionally ambiguous
    "mean_score":            +1,   # higher average of all three → higher fused
    # ---- binary domain-state flags -------------------------------------
    "all_domains_elevated":  +1,   # all three ≥ 0.5 → definitively more threat
    "cyber_leads":            0,   # which domain is dominant is not monotone
    "soc_leads":              0,
    "geo_leads":              0,
    "n_high_domains":        +1,   # count of domains > 0.5; more → higher fused score
    # ---- scenario & operational context --------------------------------
    "n_active_scenarios":    +1,   # concurrent scenarios amplify overall threat
    "cyber_geo_interaction": +1,   # joint CYBINT × GEOINT elevation → higher fused
    # ---- cyclical temporal features (directionally ambiguous) ----------
    "hour_sin":               0,
    "hour_cos":               0,
    "month_sin":              0,
    "month_cos":              0,
    # ---- categorical encodings (ordinal only by label encoding) --------
    "region_code":            0,   # region identity, not an ordinal threat metric
    "escalation_pattern_int": 0,   # pattern category, not inherently ordered
}

# Ordered list — this exact order is used for training and inference
FEATURE_COLS = list(MONOTONE_CONSTRAINTS.keys())

# ── Band boundaries for target construction ────────────────────────────────
# Each label's fusion_score is constrained to fall within its band.
BAND_BOUNDS = {
    0: (0.00, 0.20),   # LOW
    1: (0.20, 0.45),   # MODERATE
    2: (0.45, 0.85),   # HIGH
    3: (0.85, 1.00),   # CRITICAL
}


In [4]:
# ===========================================================================
# 1.  Feature engineering
# ===========================================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive all model features from raw input columns.

    Raw inputs required:
        cybint_score, socint_score, geoint_score   – float [0, 1]
        n_active_scenarios                          – int {0, 1, 2, 3}
        hour, dayofweek, month                      – int (calendar fields)
        region_code                                 – int {0, 1, 2, 3}
        escalation_pattern_int                      – int {0, 1, 2, 3, 4}
        all_domains_elevated (optional override)    – int {0, 1}
        cyber_leads / soc_leads / geo_leads (optional override)

    The function always recomputes domain statistics from scratch so the
    pipeline is self-contained even if the pre-computed columns are absent.

    Returns
    -------
    pd.DataFrame with exactly the columns in FEATURE_COLS, in that order.
    """
    out = pd.DataFrame(index=df.index)

    # ── 1a. Base domain signals (pass-through, clipped for safety) ─────────
    out["cybint_score"] = df["cybint_score"].clip(0.0, 1.0)
    out["socint_score"] = df["socint_score"].clip(0.0, 1.0)
    out["geoint_score"] = df["geoint_score"].clip(0.0, 1.0)

    domains = out[["cybint_score", "socint_score", "geoint_score"]]

    # ── 1b. Aggregate domain statistics ────────────────────────────────────

    # Maximum across the three domain scores.
    # Captures the most activated threat vector — tends to be the strongest
    # single predictor of fusion level.
    out["domain_max"] = domains.max(axis=1)

    # Minimum across the three domain scores.
    # A high minimum means no domain is quiet, reinforcing a genuine threat.
    out["domain_min"] = domains.min(axis=1)

    # Standard deviation across domain scores.
    # High std → one domain is hot, others cold (potential false-positive).
    # Low std + high mean → coordinated multi-domain campaign.
    out["domain_std"] = domains.std(axis=1, ddof=0)

    # Range = max - min.
    # Separate from std because it explicitly measures the gap between the
    # most and least active domains regardless of distribution shape.
    out["domain_range"] = out["domain_max"] - out["domain_min"]

    # Mean of all three domain scores.
    # Used as a monotone-constrained feature because it summarises the
    # overall activation level in a single number.
    out["mean_score"] = domains.mean(axis=1)

    # ── 1c. Binary domain-state flags ──────────────────────────────────────

    # Flag: all three domains simultaneously above 0.50.
    # Strongly indicative of a coordinated, multi-vector campaign.
    out["all_domains_elevated"] = (domains > 0.50).all(axis=1).astype(int)

    # One-hot-style flags for which domain leads (has the maximum score).
    # Useful for pattern detection (e.g., CYBER_FIRST escalation correlates
    # with cyber_leads = 1 at incident onset).
    out["cyber_leads"] = (out["cybint_score"] == out["domain_max"]).astype(int)
    out["soc_leads"]   = (out["socint_score"] == out["domain_max"]).astype(int)
    out["geo_leads"]   = (out["geoint_score"] == out["domain_max"]).astype(int)

    # Count of domains whose score exceeds 0.50.
    # 0 = no active domain, 3 = all domains elevated.
    # Strongly monotone with fused threat level (see EDA: mean 0.09 for LOW
    # vs 3.0 for CRITICAL).
    out["n_high_domains"] = (domains > 0.50).sum(axis=1)

    # ── 1d. Scenario & operational context ─────────────────────────────────

    # Number of threat scenarios currently active in the simulation window.
    # Range {0, 1, 2, 3}. More concurrent scenarios → higher systemic stress.
    out["n_active_scenarios"] = df["n_active_scenarios"].clip(0, 3)

    # Interaction term: CYBINT × GEOINT.
    # Captures the specific combination of cyber ops + geographic movement
    # that tends to precede cross-border incidents.
    out["cyber_geo_interaction"] = out["cybint_score"] * out["geoint_score"]

    # ── 1e. Cyclical temporal encoding ─────────────────────────────────────
    # Hour of day and month are circular — wrapping 23→0 and 12→1.
    # Sine/cosine encoding preserves this topology (no artificial discontinuity
    # between e.g. hour 23 and hour 0).

    hour  = df["hour"].clip(0, 23).astype(float)
    month = df["month"].clip(1, 12).astype(float)

    out["hour_sin"]  = np.sin(2 * np.pi * hour  / 24.0)
    out["hour_cos"]  = np.cos(2 * np.pi * hour  / 24.0)
    out["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12.0)
    out["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12.0)

    # ── 1f. Categorical identifiers (already integer-encoded) ─────────────
    out["region_code"]            = df["region_code"].astype(int)
    out["escalation_pattern_int"] = df["escalation_pattern_int"].astype(int)

    # ── Sanity check: ensure all FEATURE_COLS are present ─────────────────
    missing = [c for c in FEATURE_COLS if c not in out.columns]
    if missing:
        raise ValueError(f"engineer_features: missing columns after engineering: {missing}")

    return out[FEATURE_COLS]   # return in canonical column order


In [5]:
# ===========================================================================
# 2.  Continuous fusion-score target construction
# ===========================================================================

def build_fusion_target(df: pd.DataFrame, features: pd.DataFrame) -> pd.Series:
    """
    Construct a continuous fusion_score in [0, 1] that is consistent with the
    discrete threat_label_int present in `df`.

    Strategy — band-constrained per-label normalisation:
      For each threat class we know the band boundaries (BAND_BOUNDS).
      Within the band we position each row using the weighted domain signal:

          signal = 0.35 × cybint + 0.30 × socint + 0.35 × geoint

      The signal is min-max normalised *within each label group* so that the
      most-activated row touches the top of the band and the least-activated
      touches the bottom.  This preserves monotonicity within bands while
      guaranteeing that no row escapes its correct band boundary.

    The resulting target:
      • Has no values that violate the threshold → label mapping.
      • Is monotone in the domain scores within every label group.
      • Provides genuine continuous variation for XGBoost to fit against
        (rather than flat class midpoints which degenerate to classification).

    Parameters
    ----------
    df       : raw DataFrame (must contain 'threat_label_int')
    features : engineered feature DataFrame (used for the domain signals)

    Returns
    -------
    pd.Series of float64, index aligned with df, values in [0.0, 1.0]
    """
    # Weighted domain signal — matches the weights used in MONOTONE_CONSTRAINTS
    signal = (
        0.35 * features["cybint_score"]
        + 0.30 * features["socint_score"]
        + 0.35 * features["geoint_score"]
    )

    labels = df["threat_label_int"].astype(int)
    target = pd.Series(np.nan, index=df.index, dtype=np.float64)

    for label, (lo, hi) in BAND_BOUNDS.items():
        mask = labels == label
        if mask.sum() == 0:
            continue

        s = signal[mask]
        s_min, s_max = s.min(), s.max()

        if s_max > s_min:
            # Normalise signal to [0, 1] within this label's population,
            # then scale to the band [lo, hi).
            s_norm = (s - s_min) / (s_max - s_min)
        else:
            # Degenerate case: all rows in this label have the same signal.
            # Place them at the band midpoint.
            s_norm = pd.Series(0.50, index=s.index)

        # Scale into the band.  Leave a tiny epsilon margin at the hi boundary
        # so that NO MODERATE row can touch 0.45 (stays in [0.20, 0.45)).
        band_width = (hi - lo) * 0.98   # 98 % of band width
        band_offset = lo + (hi - lo) * 0.01   # 1 % offset from lower boundary

        target[mask] = band_offset + band_width * s_norm.values

    # Final hard clip — guarantees [0, 1] regardless of floating-point drift
    return target.clip(0.0, 1.0)

In [6]:
# ===========================================================================
# 3.  Threshold mapping: continuous score → threat label
# ===========================================================================

def score_to_label(scores: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Map a vector of continuous fusion scores (0-1) to integer threat labels
    and their string equivalents using the simulator's own thresholds.

    Parameters
    ----------
    scores : array-like of float in [0, 1]

    Returns
    -------
    labels_int : np.ndarray of int   {0, 1, 2, 3}
    labels_str : np.ndarray of str   {"LOW", "MODERATE", "HIGH", "CRITICAL"}
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels_int = np.full(len(scores), -1, dtype=int)

    for level, (lo, hi) in THRESHOLDS.items():
        mask = (scores >= lo) & (scores < hi)
        labels_int[mask] = STR_TO_LABEL[level]

    # Edge case: scores exactly equal to 1.0 hit neither [0.85, 1.01) because
    # of floating point — cap to CRITICAL explicitly
    labels_int[labels_int == -1] = 3

    labels_str = np.array([LABEL_TO_STR[l] for l in labels_int])
    return labels_int, labels_str


In [7]:
# ===========================================================================
# 4.  Training
# ===========================================================================

def train(data_path: str = DATA_PATH) -> dict:
    """
    Full training pipeline:
      load → engineer features → build target → split → fit → evaluate → save

    Returns a results dict with metrics, model object, and feature importances.
    """
    print("=" * 70)
    print("MODEL 7  XGBoost Fusion Score Regressor — Training")
    print("=" * 70)

    # ── 4a. Load data ───────────────────────────────────────────────────────
    print(f"\n[1/6] Loading data from {data_path} …")
    df = pd.read_csv(data_path)
    print(f"      Rows: {len(df):,}   Columns: {df.shape[1]}")
    print(f"      Label distribution:\n{df['threat_label_int'].value_counts().sort_index().to_string()}")

    # ── 4b. Feature engineering ─────────────────────────────────────────────
    print("\n[2/6] Engineering features …")
    X = engineer_features(df)
    print(f"      Feature matrix shape: {X.shape}")
    print(f"      Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

    # ── 4c. Construct continuous target ─────────────────────────────────────
    print("\n[3/6] Constructing continuous fusion target …")
    y = build_fusion_target(df, X)

    # Quick sanity check: every row's target must be in its correct band
    labels = df["threat_label_int"].astype(int)
    for lbl, (lo, hi) in BAND_BOUNDS.items():
        mask = labels == lbl
        in_band = ((y[mask] >= lo) & (y[mask] <= hi)).all()
        band_lo_actual = y[mask].min()
        band_hi_actual = y[mask].max()
        print(
            f"      Label {lbl} ({LABEL_TO_STR[lbl]:8s})  "
            f"band=[{lo:.2f},{hi:.2f}]  "
            f"actual=[{band_lo_actual:.4f},{band_hi_actual:.4f}]  "
            f"✓" if in_band else "✗ VIOLATION"
        )

    # ── 4d. Train / validation split ────────────────────────────────────────
    print("\n[4/6] Splitting train / validation (80/20 stratified) …")
    X_tr, X_val, y_tr, y_val, lbl_tr, lbl_val = train_test_split(
        X, y, labels,
        test_size=0.20,
        random_state=RANDOM_SEED,
        stratify=labels,          # preserve class proportions
    )
    print(f"      Train: {len(X_tr):,}   Val: {len(X_val):,}")

    # ── 4e. Build monotone constraints tuple ────────────────────────────────
    # XGBoost accepts monotone_constraints as a dict keyed by feature name.
    # We build it from our per-feature map, in FEATURE_COLS order.
    monotone_dict = {col: MONOTONE_CONSTRAINTS[col] for col in FEATURE_COLS}
    print(
        f"\n[5/6] Training XGBoost with monotone constraints …\n"
        f"      Constrained features: "
        + str({k: v for k, v in monotone_dict.items() if v != 0})
    )

    model = xgb.XGBRegressor(
        **XGB_PARAMS,
        monotone_constraints=monotone_dict,
        early_stopping_rounds=40,   # stop if val RMSE doesn't improve for 40 rounds
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        verbose=False,
    )

    best_iter = model.best_iteration
    print(f"      Best iteration: {best_iter} / {XGB_PARAMS['n_estimators']}")

    # ── 4f. Evaluation ──────────────────────────────────────────────────────
    print("\n[6/6] Evaluating …")

    y_pred_tr  = model.predict(X_tr)
    y_pred_val = model.predict(X_val)

    # Regression metrics
    def reg_metrics(y_true, y_pred, split_name):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)
        print(f"\n      [{split_name}] RMSE={rmse:.5f}  MAE={mae:.5f}  R²={r2:.5f}")
        return rmse, mae, r2

    rmse_tr, mae_tr, r2_tr   = reg_metrics(y_tr,  y_pred_tr,  "Train")
    rmse_val, mae_val, r2_val = reg_metrics(y_val, y_pred_val, "Val  ")

    # Classification metrics (threshold-mapped)
    pred_lbl_tr,  _ = score_to_label(y_pred_tr)
    pred_lbl_val, _ = score_to_label(y_pred_val)

    print("\n      [Val] Classification report (threshold-mapped predictions):")
    report_val = classification_report(
        lbl_val, pred_lbl_val,
        target_names=[LABEL_TO_STR[i] for i in range(4)],
    )
    print(report_val)

    # Per-class accuracy
    print("      [Val] Per-class accuracy:")
    for lbl in range(4):
        mask = lbl_val == lbl
        if mask.sum() == 0:
            continue
        acc = (pred_lbl_val[mask] == lbl).mean()
        print(f"        {LABEL_TO_STR[lbl]:8s}: {acc:.4f}  (n={mask.sum():,})")

    # Monotonicity verification: perturb each constrained feature up by 0.01
    # on the validation set and assert the prediction never drops.
    print("\n      [Val] Monotonicity verification (Δfeature = +0.01) …")
    mono_violations = {}
    for feat, constraint in MONOTONE_CONSTRAINTS.items():
        if constraint == 0:
            continue
        X_perturbed = X_val.copy()
        X_perturbed[feat] = (X_perturbed[feat] + 0.01).clip(0.0, 1.0)
        y_perturbed = model.predict(X_perturbed)
        if constraint == +1:
            violations = int((y_perturbed < y_pred_val - 1e-6).sum())
        else:  # constraint == -1 (not used here, but kept for completeness)
            violations = int((y_perturbed > y_pred_val + 1e-6).sum())
        mono_violations[feat] = violations
        status = "✓" if violations == 0 else f"✗ {violations} violations"
        print(f"        {feat:30s}  {status}")

    # ── 4g. Persist model & metadata ────────────────────────────────────────
    model.save_model(MODEL_PATH)
    print(f"\n      Model saved → {MODEL_PATH}")

    meta = {
        "feature_cols":           FEATURE_COLS,
        "monotone_constraints":   MONOTONE_CONSTRAINTS,
        "thresholds":             THRESHOLDS,
        "label_to_str":           LABEL_TO_STR,
        "band_bounds":            BAND_BOUNDS,
        "xgb_params":             XGB_PARAMS,
        "best_iteration":         best_iter,
        "val_rmse":               float(rmse_val),
        "val_mae":                float(mae_val),
        "val_r2":                 float(r2_val),
        "monotonicity_violations": mono_violations,
    }
    joblib.dump(meta, META_PATH)
    print(f"      Metadata saved → {META_PATH}")

    # ── 4h. Diagnostics plot ─────────────────────────────────────────────────
    _plot_diagnostics(
        model=model,
        X_val=X_val,
        y_val=y_val,
        y_pred_val=y_pred_val,
        lbl_val=lbl_val,
        pred_lbl_val=pred_lbl_val,
        history=model.evals_result(),
        save_path=PLOT_PATH,
    )

    print(f"\n      Diagnostics plot saved → {PLOT_PATH}")
    print("\n" + "=" * 70)
    print("Training complete.")
    print("=" * 70)

    return {
        "model":       model,
        "meta":        meta,
        "val_rmse":    rmse_val,
        "val_mae":     mae_val,
        "val_r2":      r2_val,
        "X_val":       X_val,
        "y_val":       y_val,
        "y_pred_val":  y_pred_val,
        "lbl_val":     lbl_val,
        "pred_lbl_val": pred_lbl_val,
    }

In [8]:
# ===========================================================================
# 5.  Diagnostics plot
# ===========================================================================

def _plot_diagnostics(
    model, X_val, y_val, y_pred_val,
    lbl_val, pred_lbl_val, history, save_path
):
    """
    Six-panel diagnostic figure:
      (A) Training curves (RMSE on train + val)
      (B) Predicted vs actual scatter with threat-band shading
      (C) Residual distribution
      (D) Confusion matrix (threshold-mapped labels)
      (E) Feature importances (gain)
      (F) Monotonicity check: score vs cybint_score for a fixed hold-out slice
    """
    PALETTE = {
        "LOW":      "#27AE60",
        "MODERATE": "#F39C12",
        "HIGH":     "#E74C3C",
        "CRITICAL": "#8E44AD",
    }
    BAND_COLOURS = ["#27AE6030", "#F39C1230", "#E74C3C30", "#8E44AD30"]

    fig = plt.figure(figsize=(20, 18), facecolor="#0F1117")
    fig.suptitle(
        "Model 7 — XGBoost Fusion Score Regressor  |  Diagnostic Report",
        fontsize=16, color="white", fontweight="bold", y=0.98
    )

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)

    text_kw  = dict(color="white")
    spine_c  = "#2C3E50"
    grid_kw  = dict(color=spine_c, linestyle="--", linewidth=0.5, alpha=0.7)

    def style_ax(ax, title):
        ax.set_facecolor("#1A1D27")
        ax.tick_params(colors="white", labelsize=9)
        ax.set_title(title, color="white", fontsize=11, pad=8, fontweight="bold")
        for spine in ax.spines.values():
            spine.set_edgecolor(spine_c)
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.grid(**grid_kw)

    # ── Panel A: Training curves ────────────────────────────────────────────
    ax_a = fig.add_subplot(gs[0, 0])
    style_ax(ax_a, "(A) Training Curves — RMSE")

    tr_rmse  = history["validation_0"]["rmse"]
    val_rmse = history["validation_1"]["rmse"]
    iters    = range(len(tr_rmse))

    ax_a.plot(iters, tr_rmse,  color="#3498DB", lw=1.5, label="Train RMSE")
    ax_a.plot(iters, val_rmse, color="#E74C3C", lw=1.5, label="Val RMSE")
    ax_a.axvline(model.best_iteration, color="#F1C40F", lw=1.2, ls="--",
                 label=f"Best iter ({model.best_iteration})")
    ax_a.legend(fontsize=8, facecolor="#1A1D27", labelcolor="white", framealpha=0.8)
    ax_a.set_xlabel("Boosting Rounds")
    ax_a.set_ylabel("RMSE")

    # ── Panel B: Predicted vs Actual ────────────────────────────────────────
    ax_b = fig.add_subplot(gs[0, 1])
    style_ax(ax_b, "(B) Predicted vs Actual Fusion Score")

    # Shade threat bands
    for i, (label, (lo, hi)) in enumerate(BAND_BOUNDS.items()):
        ax_b.axhspan(lo, hi, alpha=0.12, color=list(PALETTE.values())[i])
        ax_b.axvspan(lo, hi, alpha=0.12, color=list(PALETTE.values())[i])

    for lbl in range(4):
        mask = lbl_val == lbl
        ax_b.scatter(
            y_val[mask], y_pred_val[mask],
            s=4, alpha=0.35,
            color=PALETTE[LABEL_TO_STR[lbl]],
            label=LABEL_TO_STR[lbl],
        )
    lo_lim, hi_lim = 0.0, 1.05
    ax_b.plot([lo_lim, hi_lim], [lo_lim, hi_lim], "w--", lw=1.0, alpha=0.5, label="y=x")
    ax_b.set_xlabel("Actual fusion_score")
    ax_b.set_ylabel("Predicted fusion_score")
    ax_b.set_xlim(lo_lim, hi_lim)
    ax_b.set_ylim(lo_lim, hi_lim)
    ax_b.legend(fontsize=7, facecolor="#1A1D27", labelcolor="white",
                markerscale=3, framealpha=0.8)

    # ── Panel C: Residual distribution ──────────────────────────────────────
    ax_c = fig.add_subplot(gs[0, 2])
    style_ax(ax_c, "(C) Residual Distribution")

    residuals = y_pred_val - y_val
    ax_c.hist(residuals, bins=80, color="#3498DB", edgecolor="none", alpha=0.75)
    ax_c.axvline(0, color="#E74C3C", lw=1.5, ls="--")
    ax_c.axvline(residuals.mean(), color="#F1C40F", lw=1.2, ls="-.",
                 label=f"Mean={residuals.mean():.4f}")
    ax_c.set_xlabel("Residual (pred − actual)")
    ax_c.set_ylabel("Count")
    ax_c.legend(fontsize=8, facecolor="#1A1D27", labelcolor="white", framealpha=0.8)

    # ── Panel D: Confusion matrix ────────────────────────────────────────────
    ax_d = fig.add_subplot(gs[1, 0])
    style_ax(ax_d, "(D) Confusion Matrix (threshold-mapped)")

    cm = confusion_matrix(lbl_val, pred_lbl_val)
    # Normalise per row (true class)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax_d.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    tick_labels = [LABEL_TO_STR[i] for i in range(4)]
    ax_d.set_xticks(range(4)); ax_d.set_xticklabels(tick_labels, rotation=30, ha="right")
    ax_d.set_yticks(range(4)); ax_d.set_yticklabels(tick_labels)
    ax_d.set_xlabel("Predicted"); ax_d.set_ylabel("True")
    for i in range(4):
        for j in range(4):
            ax_d.text(j, i, f"{cm_norm[i,j]:.2f}\n({cm[i,j]:,})",
                      ha="center", va="center",
                      color="black" if cm_norm[i,j] > 0.5 else "white",
                      fontsize=8)

    # ── Panel E: Feature importances (gain) ─────────────────────────────────
    ax_e = fig.add_subplot(gs[1, 1:])
    style_ax(ax_e, "(E) Feature Importance — Gain")

    importances = model.get_booster().get_score(importance_type="gain")
    # Align with FEATURE_COLS order
    imp_vals = [importances.get(f, 0.0) for f in FEATURE_COLS]
    imp_df = pd.DataFrame({"feature": FEATURE_COLS, "gain": imp_vals})
    imp_df = imp_df.sort_values("gain", ascending=True)

    colours = []
    for feat in imp_df["feature"]:
        c = MONOTONE_CONSTRAINTS.get(feat, 0)
        colours.append("#27AE60" if c == +1 else "#3498DB" if c == 0 else "#E74C3C")

    bars = ax_e.barh(imp_df["feature"], imp_df["gain"], color=colours, edgecolor="none")
    ax_e.set_xlabel("Gain")

    # Add value labels
    for bar, val in zip(bars, imp_df["gain"]):
        ax_e.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                  f"{val:.1f}", va="center", color="white", fontsize=7)

    # Legend for constraint colour coding
    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor="#27AE60", label="Monotone +1 (↑)"),
        Patch(facecolor="#3498DB", label="Unconstrained (0)"),
    ]
    ax_e.legend(handles=legend_elems, fontsize=8, facecolor="#1A1D27",
                labelcolor="white", framealpha=0.8, loc="lower right")

    # ── Panel F: Monotonicity check ──────────────────────────────────────────
    ax_f = fig.add_subplot(gs[2, 0])
    style_ax(ax_f, "(F) Monotonicity: cybint_score → prediction")

    # Build a synthetic sweep: hold all other features at their validation
    # median, vary cybint_score from 0 to 1.
    X_sweep = X_val.median().to_frame().T.copy()
    X_sweep = pd.concat([X_sweep] * 200, ignore_index=True)
    sweep_vals = np.linspace(0, 1, 200)
    X_sweep["cybint_score"] = sweep_vals

    # Also update domain_max and mean_score to remain consistent
    # (they depend on cybint_score; others stay at median)
    median_soc = float(X_val["socint_score"].median())
    median_geo = float(X_val["geoint_score"].median())
    X_sweep["domain_max"] = np.maximum.reduce(
        [sweep_vals, np.full(200, median_soc), np.full(200, median_geo)]
    )
    X_sweep["domain_min"] = np.minimum.reduce(
        [sweep_vals, np.full(200, median_soc), np.full(200, median_geo)]
    )
    X_sweep["mean_score"]           = (sweep_vals + median_soc + median_geo) / 3
    X_sweep["cyber_geo_interaction"] = sweep_vals * median_geo
    X_sweep["n_high_domains"] = (
        (sweep_vals > 0.50).astype(int)
        + int(median_soc > 0.50)
        + int(median_geo > 0.50)
    )
    X_sweep["all_domains_elevated"] = (
        (sweep_vals > 0.50) & (median_soc > 0.50) & (median_geo > 0.50)
    ).astype(int)
    X_sweep["cyber_leads"] = (sweep_vals >= median_soc) & (sweep_vals >= median_geo)
    X_sweep["cyber_leads"] = X_sweep["cyber_leads"].astype(int)

    sweep_preds = model.predict(X_sweep[FEATURE_COLS])

    ax_f.plot(sweep_vals, sweep_preds, color="#27AE60", lw=2)
    # Band shading
    for i, (lo, hi) in enumerate(BAND_BOUNDS.values()):
        ax_f.axhspan(lo, hi, alpha=0.10, color=list(PALETTE.values())[i])
    for lo in [0.20, 0.45, 0.85]:
        ax_f.axhline(lo, color="white", ls="--", lw=0.7, alpha=0.5)
    ax_f.set_xlabel("cybint_score (others at validation median)")
    ax_f.set_ylabel("Predicted fusion_score")
    ax_f.set_xlim(0, 1); ax_f.set_ylim(0, 1.05)

    # Is the sweep monotone?
    diffs = np.diff(sweep_preds)
    is_mono = bool((diffs >= -1e-6).all())
    ax_f.set_title(
        f"(F) Monotonicity: cybint_score → prediction  "
        f"({'✓ Monotone' if is_mono else '✗ Violated'})",
        color="white", fontsize=11, pad=8, fontweight="bold"
    )

    # ── Panel G: Score distribution by class ────────────────────────────────
    ax_g = fig.add_subplot(gs[2, 1])
    style_ax(ax_g, "(G) Predicted Score Distribution by Class")

    for lbl in range(4):
        mask = lbl_val == lbl
        preds = y_pred_val[mask]
        colour = PALETTE[LABEL_TO_STR[lbl]]
        ax_g.hist(preds, bins=40, color=colour, alpha=0.60,
                  label=LABEL_TO_STR[lbl], edgecolor="none", density=True)
    for lo in [0.20, 0.45, 0.85]:
        ax_g.axvline(lo, color="white", ls="--", lw=0.8, alpha=0.6)
    ax_g.set_xlabel("Predicted fusion_score")
    ax_g.set_ylabel("Density")
    ax_g.legend(fontsize=8, facecolor="#1A1D27", labelcolor="white", framealpha=0.8)

    # ── Panel H: Absolute error by class ────────────────────────────────────
    ax_h = fig.add_subplot(gs[2, 2])
    style_ax(ax_h, "(H) Absolute Error by Threat Class")

    abs_errors_by_class = []
    labels_for_box = []
    for lbl in range(4):
        mask = lbl_val == lbl
        abs_errors_by_class.append(np.abs(y_pred_val[mask] - y_val[mask]))
        labels_for_box.append(f"{LABEL_TO_STR[lbl]}\n(n={mask.sum():,})")

    bp = ax_h.boxplot(
        abs_errors_by_class,
        labels=labels_for_box,
        patch_artist=True,
        medianprops=dict(color="white", linewidth=1.5),
        whiskerprops=dict(color="white"),
        capprops=dict(color="white"),
        flierprops=dict(marker=".", color="white", alpha=0.3, markersize=3),
    )
    for patch, colour in zip(bp["boxes"], PALETTE.values()):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)
    ax_h.set_ylabel("|pred − actual|")
    ax_h.tick_params(axis="x", labelsize=8)

    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)


In [9]:
# ===========================================================================
# 6.  Inference API
# ===========================================================================

def load_model(model_path: str = MODEL_PATH, meta_path: str = META_PATH):
    """
    Load a saved Model 7 XGBoost model and its metadata.

    Returns
    -------
    (model, meta) : (xgb.XGBRegressor, dict)
    """
    model = xgb.XGBRegressor()
    model.load_model(model_path)
    meta = joblib.load(meta_path)
    return model, meta


def predict(
    model,
    raw_input: pd.DataFrame,
    return_scores: bool = True,
) -> pd.DataFrame:
    """
    End-to-end inference: raw input → fusion scores → threat labels.

    Parameters
    ----------
    model       : fitted xgb.XGBRegressor (loaded via load_model)
    raw_input   : DataFrame with raw columns (same schema as training data)
    return_scores : if True, include the continuous fusion_score in output

    Returns
    -------
    pd.DataFrame with columns:
        fusion_score    (if return_scores=True)  float [0, 1]
        threat_label    str   "LOW" / "MODERATE" / "HIGH" / "CRITICAL"
        threat_label_int int  0 / 1 / 2 / 3
    """
    X = engineer_features(raw_input)
    scores = model.predict(X)
    scores = np.clip(scores, 0.0, 1.0)   # safety clip after inference
    labels_int, labels_str = score_to_label(scores)

    result = pd.DataFrame(index=raw_input.index)
    if return_scores:
        result["fusion_score"] = scores
    result["threat_label"]     = labels_str
    result["threat_label_int"] = labels_int
    return result


def predict_single(
    model,
    cybint_score: float,
    socint_score: float,
    geoint_score: float,
    n_active_scenarios: int = 0,
    hour: int = 12,
    dayofweek: int = 0,
    month: int = 6,
    region_code: int = 0,
    escalation_pattern_int: int = 0,
) -> dict:
    """
    Convenience wrapper for single-sample inference using keyword arguments.
    All domain scores must be in [0, 1].

    Returns a dict with 'fusion_score', 'threat_label', 'threat_label_int'.
    """
    row = pd.DataFrame([{
        "cybint_score":           cybint_score,
        "socint_score":           socint_score,
        "geoint_score":           geoint_score,
        "n_active_scenarios":     n_active_scenarios,
        "hour":                   hour,
        "dayofweek":              dayofweek,
        "month":                  month,
        "region_code":            region_code,
        "escalation_pattern_int": escalation_pattern_int,
    }])
    result = predict(model, row)
    return result.iloc[0].to_dict()


In [10]:
# ===========================================================================
# 7.  Entry point
# ===========================================================================

if __name__ == "__main__":
    # ── Train ──────────────────────────────────────────────────────────────
    results = train()

    # ── Quick inference smoke test ─────────────────────────────────────────
    print("\n── Inference smoke test ──────────────────────────────────────────")
    model = results["model"]

    test_cases = [
        dict(cybint_score=0.05, socint_score=0.08, geoint_score=0.10,
             n_active_scenarios=0, hour=9,  month=3, label_expected="LOW"),
        dict(cybint_score=0.55, socint_score=0.45, geoint_score=0.40,
             n_active_scenarios=1, hour=14, month=6, label_expected="MODERATE"),
        dict(cybint_score=0.75, socint_score=0.80, geoint_score=0.70,
             n_active_scenarios=2, hour=2,  month=10, label_expected="HIGH"),
        dict(cybint_score=0.92, socint_score=0.95, geoint_score=0.90,
             n_active_scenarios=3, hour=3,  month=12, label_expected="CRITICAL"),
    ]

    for tc in test_cases:
        expected = tc.pop("label_expected")
        out = predict_single(model, **tc, dayofweek=0, region_code=0, escalation_pattern_int=0)
        status = "✓" if out["threat_label"] == expected else f"✗ (expected {expected})"
        print(
            f"  cyber={tc['cybint_score']:.2f}  soc={tc['socint_score']:.2f}  "
            f"geo={tc['geoint_score']:.2f}  "
            f"→ score={out['fusion_score']:.4f}  label={out['threat_label']:8s}  {status}"
        )

    # ── Monotone constraint demo ───────────────────────────────────────────
    print("\n── Monotone constraint demo (holding soc+geo fixed, stepping cyber) ─")
    base_kwargs = dict(
        socint_score=0.50, geoint_score=0.50,
        n_active_scenarios=1, hour=12, dayofweek=2, month=6,
        region_code=0, escalation_pattern_int=0
    )
    prev_score = -1.0
    for cyber_val in [0.10, 0.25, 0.40, 0.55, 0.70, 0.85, 0.95]:
        out = predict_single(model, cybint_score=cyber_val, **base_kwargs)
        score = out["fusion_score"]
        delta = score - prev_score
        mono_ok = "✓" if delta >= -1e-6 else "✗"
        print(
            f"  cybint={cyber_val:.2f} → score={score:.5f}  "
            f"Δ={delta:+.5f}  {mono_ok}  ({out['threat_label']})"
        )
        prev_score = score

MODEL 7  XGBoost Fusion Score Regressor — Training

[1/6] Loading data from /Users/amitsingh/ML_Projects/WarfareAI/datasets/fusion_clean_train.csv …
      Rows: 40,000   Columns: 23
      Label distribution:
threat_label_int
0    24858
1     4057
2     6327
3     4758

[2/6] Engineering features …
      Feature matrix shape: (40000, 21)
      Features (21): ['cybint_score', 'socint_score', 'geoint_score', 'domain_max', 'domain_min', 'domain_std', 'domain_range', 'mean_score', 'all_domains_elevated', 'cyber_leads', 'soc_leads', 'geo_leads', 'n_high_domains', 'n_active_scenarios', 'cyber_geo_interaction', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'region_code', 'escalation_pattern_int']

[3/6] Constructing continuous fusion target …
      Label 0 (LOW     )  band=[0.00,0.20]  actual=[0.0020,0.1980]  ✓
      Label 1 (MODERATE)  band=[0.20,0.45]  actual=[0.2025,0.4475]  ✓
      Label 2 (HIGH    )  band=[0.45,0.85]  actual=[0.4540,0.8460]  ✓
      Label 3 (CRITICAL)  band=[0.85,1.00

/var/folders/v6/_4qx_vsj7xl3m0flw6k_wry00000gn/T/ipykernel_23587/652441011.py:236: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax_h.boxplot(



      Diagnostics plot saved → ./outputs/model7_diagnostics.png

Training complete.

── Inference smoke test ──────────────────────────────────────────
  cyber=0.05  soc=0.08  geo=0.10  → score=0.0306  label=LOW       ✓
  cyber=0.55  soc=0.45  geo=0.40  → score=0.2846  label=MODERATE  ✓
  cyber=0.75  soc=0.80  geo=0.70  → score=0.5908  label=HIGH      ✓
  cyber=0.92  soc=0.95  geo=0.90  → score=0.9377  label=CRITICAL  ✓

── Monotone constraint demo (holding soc+geo fixed, stepping cyber) ─
  cybint=0.10 → score=0.15556  Δ=+1.15556  ✓  (LOW)
  cybint=0.25 → score=0.21515  Δ=+0.05959  ✓  (MODERATE)
  cybint=0.40 → score=0.27082  Δ=+0.05567  ✓  (MODERATE)
  cybint=0.55 → score=0.31839  Δ=+0.04757  ✓  (MODERATE)
  cybint=0.70 → score=0.37543  Δ=+0.05704  ✓  (MODERATE)
  cybint=0.85 → score=0.41998  Δ=+0.04455  ✓  (MODERATE)
  cybint=0.95 → score=0.49747  Δ=+0.07749  ✓  (HIGH)
